# Particle Degeneracy & Resampling

Welcome to the seventh interactive notebook of the `digital-twins` repository.

We introduce a fatal flaw in the basic Sequential Importance Sampling (SIS) Particle Filter: **The Degeneracy Phenomenon**.

As a particle filter progresses through time, we multiply each particle's old weight by its new measurement likelihood. Because we are multiplying probabilities (numbers between 0 and 1) step after step, the variance of the weights strictly increases. After a few steps, exactly **one particle** will end up with a weight of `1.0`, and all other particles will have a weight of `0.0000000`.

The filter becomes useless because it is only tracking a single hypothesis. 

### The Solution: Resampling (Algorithm 6.3)
To fix this, we introduce **Resampling**. We monitor the health of the particles, and when they start to degenerate, we pause. We "kill" the low-weight particles, "clone" the high-weight particles, and reset all their weights back to $\frac{1}{N}$.

### Tracking Health: Effective Sample Size ($N_{eff}$)
In practice, we measure degeneracy using the Effective Sample Size:
$$ N_{eff} = \frac{1}{\sum_{i=1}^{N} (w_k^{(i)})^2} $$
If $N_{eff}$ drops significantly below $N$ (the total number of particles), the filter is dying.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, Checkbox

### Interactive Learning: Watch the Filter Die

We will simulate a 1D mobile agent starting at `x = 50` moving to the right. We initialize $N=200$ particles.

**Instructions:**
1. **Keep Resampling `OFF`**: Drag the `Time Step` slider slowly to the right. Watch the top graph (Particle Weights). By Step 5 or 6, notice how all the blue dots flatline at 0, and a single dot shoots up to 1.0. Look at the bottom graph: the Effective Sample Size crashes to 1. The filter is dead.
2. **Turn Resampling `ON`**: Check the box and drag the slider again. Whenever $N_{eff}$ drops too low, the algorithm clones the good particles and resets their weights. The filter stays healthy forever!

In [ ]:
def interactive_degeneracy(time_steps=1, enable_resampling=False):
    # --- Fixed Seed for Reproducible Reality ---
    np.random.seed(42)
    
    # --- Parameters ---
    N = 200
    sensor_noise_std = 3.0
    process_noise_std = 1.5
    vel = 5.0
    
    # --- Initialization ---
    true_pos = 50.0
    particles = np.random.uniform(30, 70, N)
    weights = np.ones(N) / N
    
    neff_history =[N]  # Starts at perfect health
    
    # --- Simulation Loop ---
    for k in range(1, time_steps + 1):
        # 1. Reality moves
        true_pos += vel + np.random.normal(0, 1.0)
        measurement = true_pos + np.random.normal(0, sensor_noise_std)
        
        # 2. PF Predict
        particles += vel + np.random.normal(0, process_noise_std, N)
        
        # 3. PF Update Weights
        variance = sensor_noise_std ** 2
        diff = particles - measurement
        likelihoods = np.exp(-0.5 * (diff ** 2) / variance) 
        
        weights *= likelihoods
        weight_sum = np.sum(weights)
        
        if weight_sum == 0:
            weights = np.ones(N) / N
        else:
            weights /= weight_sum
            
        # 4. Calculate Health (N_eff)
        neff = 1.0 / np.sum(weights ** 2)
        neff_history.append(neff)
        
        # 5. Conditional Resampling (Algorithm 6.3)
        if enable_resampling and neff < (N / 2.0):
            # Systematic Resampling
            cum_sum = np.cumsum(weights)
            cum_sum[-1] = 1.0
            indices = np.zeros(N, dtype=int)
            r = np.random.uniform(0, 1.0 / N)
            
            i = 0
            for j in range(N):
                u = r + j / N
                while u > cum_sum[i]:
                    i += 1
                indices[j] = i
                
            particles = particles[indices]
            weights = np.ones(N) / N

    # --- Plotting ---
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), gridspec_kw={'height_ratios': [2, 1]})
    
    # Plot 1: Particle Weights at the current step
    ax1.scatter(particles, weights, c='royalblue', edgecolors='black', alpha=0.7, s=40, label='Particle Weights')
    ax1.axvline(true_pos, color='green', linestyle='--', linewidth=2, label='True Position')
    ax1.axvline(measurement, color='red', linestyle=':', linewidth=2, label='Noisy Sensor')
    
    title_status = "HEALTHY" if neff_history[-1] > (N/10) else "DEGENERATED!"
    color_status = "green" if neff_history[-1] > (N/10) else "red"
    ax1.set_title(f"Particle Weight Distribution at Step {time_steps} - Status: {title_status}", color=color_status, fontweight='bold')
    ax1.set_ylabel("Importance Weight $w^{(i)}$")
    ax1.set_xlabel("1D Position")
    ax1.set_ylim(-0.05, 1.05)
    ax1.grid(True, alpha=0.3)
    ax1.legend(loc='upper right')

    # Plot 2: N_eff History
    time_array = np.arange(0, time_steps + 1)
    ax2.plot(time_array, neff_history, 'm-o', linewidth=2, label='$N_{eff}$ (Health)')
    ax2.axhline(N / 2.0, color='orange', linestyle='--', label='Resampling Threshold ($N/2$)')
    ax2.set_title("Filter Health over Time: Effective Sample Size ($N_{eff}$)", fontweight='bold')
    ax2.set_ylabel("Effective Particles")
    ax2.set_xlabel("Time Step")
    ax2.set_ylim(0, N + 10)
    ax2.set_xlim(0, max(10, time_steps))
    ax2.grid(True, alpha=0.3)
    ax2.legend(loc='lower left')

    plt.tight_layout()
    plt.show()

# Create interactive widgets
interact(interactive_degeneracy, 
         time_steps=IntSlider(value=1, min=1, max=15, step=1, description='Time Step ($k$):', continuous_update=False, layout={'width':'500px'}),
         enable_resampling=Checkbox(value=False, description='Enable Resampling (Algorithm 6.3)', indent=False));